In [1]:
from sympleq.core.symmetries.pauli import pauli_reduce
from sympleq.core.symmetries.clifford import clifford_phase_decomposition, min_qudit_clifford_symmetry, qudit_cost
from scripts.experiments.symmetries.src.block_decomposition import block_decompose, block_decompose_optimal
from sympleq.models.Ising import ising_chain_hamiltonian, ising_2d_hamiltonian, heuristic_clifford_symmetry
from sympleq.core.circuits.gate_decomposition_to_circuit import gate_to_circuit
from sympleq.core.circuits import Gate, Circuit, gate_to_circuit
import numpy as np

In [2]:

N = 8
J = 1
h = 0.5
H = ising_chain_hamiltonian(N, J, h, periodic=True)


In [4]:
F = heuristic_clifford_symmetry(N)
# print(F.symplectic)
S, T = block_decompose_optimal(F.symplectic, 2)  # 

h_S, h_T = clifford_phase_decomposition(F.symplectic, F.phase_vector, S, T, int(H.lcm))
S_gate = Gate('S', F.qudit_indices, S, F.dimensions, h_S)
T_gate = Gate('T', F.qudit_indices, T, F.dimensions, h_T)

assert F == Circuit(F.dimensions, [T_gate.inv(), S_gate, T_gate]).composite_gate()

assert H.to_standard_form() == F.act(H).to_standard_form()
assert T_gate.act(S_gate.act(T_gate.inv().act(H))).to_standard_form() == H.to_standard_form()

assert S_gate.act(T_gate.inv().act(H)).to_standard_form() == T_gate.inv().act(H).to_standard_form()

print('Got T and S')
print('Qubit cost is ', qudit_cost(S_gate))

Got T and S
Qubit cost is  2


In [10]:
### Test F unitary

C_F = gate_to_circuit(F)
# C_S = gate_to_circuit(S)
# C_T = gate_to_circuit(T)
assert C_F.act(H).to_standard_form() == H.to_standard_form()

U_F = C_F.unitary().toarray()
H_hilbert = H.to_hilbert_space().toarray()
assert np.all(np.abs(U_F @ H_hilbert @ U_F.conj().T - H_hilbert) < 1e-8)

print('F passed')
### Test decomposition symmetry (Pauli-level)

H_prime = T_gate.inv().act(H)
H_rec = Circuit(F.dimensions, [T_gate.inv(), S_gate, T_gate]).act(H)

# For p=2 the gate_to_circuit Pauli correction drops odd phase components,
# so we verify invariance at the Pauli level using the reconstructed F.
assert H_rec.to_standard_form() == H.to_standard_form()
assert S_gate.act(H_prime).to_standard_form() == H_prime.to_standard_form()
print('F (via S,T) passed (Pauli check)')


F passed
F (via S,T) passed (Pauli check)


In [19]:
C_np = Circuit(S_gate.dimensions, C_S[0:-1])
C_S = gate_to_circuit(S_gate)

print(C_np)


NameError: name 'C_S' is not defined